In [1]:
# Cell 1: imports, configuration, reproducibility, directories

import os
import json
import random
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import f1_score

from tabgan.sampler import GANGenerator

SEED = 42
DATA_PATH = "dataset/EVSE-B-PowerCombined_filtered.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = Path("./gridsearch_runs")
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
RESULTS_DIR = BASE_DIR / "results"
OUTPUT_DIR = BASE_DIR / "output"
SYNTHETIC_DIR = OUTPUT_DIR / "synthetic"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_COLS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
CATEGORICAL_COLS = ["State"]
TARGET_COL = "Attack"

EXPECTED_STATE_VALUES = {"idle", "charging"}
EXPECTED_ATTACK_VALUES = {"syn-flood", "none", "Backdoor"}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("Device:", DEVICE)
print("Data path exists:", os.path.exists(DATA_PATH))
print("Output dir:", OUTPUT_DIR)
print("Synthetic dir:", SYNTHETIC_DIR)

Device: cpu
Data path exists: True
Output dir: gridsearch_runs/output
Synthetic dir: gridsearch_runs/output/synthetic


In [2]:
# Cell 2: load dataset and validate required columns

df = pd.read_csv(DATA_PATH)

required_cols = NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].copy()

if df.isnull().sum().sum() != 0:
    raise ValueError("Dataset contains missing values, but none were expected.")

df["State"] = df["State"].astype(str).str.strip()
df["Attack"] = df["Attack"].astype(str).str.strip()

print("Dataset shape after column selection:", df.shape)
display(df.head())

print("\nAttack distribution:")
print(df[TARGET_COL].value_counts())

print("\nState distribution:")
print(df["State"].value_counts())

Dataset shape after column selection: (49017, 6)


,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack
0,978,5.165,1027,5300,idle,syn-flood
1,872,5.161,1009,4980,idle,syn-flood
2,1017,5.165,1029,5300,idle,syn-flood
3,930,5.161,1005,5180,idle,syn-flood
4,958,5.165,1034,5180,idle,syn-flood



Attack distribution:
Attack
Backdoor     21137
none         14363
syn-flood    13517
Name: count, dtype: int64

State distribution:
State
idle        27898
charging    21119
Name: count, dtype: int64


In [3]:
# Cell 2: load dataset and validate required columns

df = pd.read_csv(DATA_PATH)

required_cols = NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].copy()

if df.isnull().sum().sum() != 0:
    raise ValueError("Dataset contains missing values, but none were expected.")

df["State"] = df["State"].astype(str).str.strip()
df["Attack"] = df["Attack"].astype(str).str.strip()

print("Dataset shape after column selection:", df.shape)
display(df.head())

print("\nAttack distribution:")
print(df[TARGET_COL].value_counts())

print("\nState distribution:")
print(df["State"].value_counts())

Dataset shape after column selection: (49017, 6)


,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack
0,978,5.165,1027,5300,idle,syn-flood
1,872,5.161,1009,4980,idle,syn-flood
2,1017,5.165,1029,5300,idle,syn-flood
3,930,5.161,1005,5180,idle,syn-flood
4,958,5.165,1034,5180,idle,syn-flood



Attack distribution:
Attack
Backdoor     21137
none         14363
syn-flood    13517
Name: count, dtype: int64

State distribution:
State
idle        27898
charging    21119
Name: count, dtype: int64


In [4]:
# Cell 3: fixed train/validation/test split
# Validation is used for grid search scoring. Test remains untouched here.

train_full_df, test_df_global = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    shuffle=True,
    stratify=df[TARGET_COL]
)

train_df_global, val_df_global = train_test_split(
    train_full_df,
    test_size=0.1765,
    random_state=SEED,
    shuffle=True,
    stratify=train_full_df[TARGET_COL]
)

print("Train:", train_df_global.shape)
print("Val:", val_df_global.shape)
print("Test:", test_df_global.shape)

Train: (34310, 6)
Val: (7354, 6)
Test: (7353, 6)


In [5]:
# Cell 3b: synthetic output sizing

target_total_rows = len(train_df_global)

target_distribution = train_df_global[TARGET_COL].value_counts(normalize=True).to_dict()

print("Target total rows:", target_total_rows)
print("Target distribution:", target_distribution)

Target total rows: 34310
Target distribution: {'Backdoor': 0.43121538909938795, 'none': 0.2930341008452346, 'syn-flood': 0.27575051005537743}


In [6]:
# Cell 4: compute target class distribution and target counts for final merged synthetic datasets

target_distribution = train_df_global[TARGET_COL].value_counts(normalize=True).to_dict()
target_counts = {k: int(target_total_rows * v) for k, v in target_distribution.items()}

remainder = target_total_rows - sum(target_counts.values())

if remainder != 0:
    largest_class = max(target_distribution, key=target_distribution.get)
    target_counts[largest_class] += remainder

print("Target distribution:", target_distribution)
print("Target counts:", target_counts)

Target distribution: {'Backdoor': 0.43121538909938795, 'none': 0.2930341008452346, 'syn-flood': 0.27575051005537743}
Target counts: {'Backdoor': 14795, 'none': 10054, 'syn-flood': 9461}


In [7]:
# Cell 5: define stage 1 coarse grid search space

stage1_param_grid = {
    "gen_x_times": [1.0, 1.5, 2.0],
    "pregeneration_frac": [2, 3],
    "is_post_process": [True],
    "bot_filter_quantile": [0.001],
    "top_filter_quantile": [0.999],
    "use_adversarial": [False, True],
    "batch_size": [120, 250],
    "patience": [10, 20],
    "epochs": [100, 200]
}

def expand_param_grid(param_grid):
    grid_keys = list(param_grid.keys())
    grid_values = list(param_grid.values())
    return [dict(zip(grid_keys, values)) for values in product(*grid_values)]

stage1_configs = expand_param_grid(stage1_param_grid)

print("Stage 1 coarse grid preview:")
for i, cfg in enumerate(stage1_configs[:5], start=1):
    print(i, cfg)

print("Total Stage 1 configurations:", len(stage1_configs))

Stage 1 coarse grid preview:
1 {'gen_x_times': 1.0, 'pregeneration_frac': 2, 'is_post_process': True, 'bot_filter_quantile': 0.001, 'top_filter_quantile': 0.999, 'use_adversarial': False, 'batch_size': 120, 'patience': 10, 'epochs': 100}
2 {'gen_x_times': 1.0, 'pregeneration_frac': 2, 'is_post_process': True, 'bot_filter_quantile': 0.001, 'top_filter_quantile': 0.999, 'use_adversarial': False, 'batch_size': 120, 'patience': 10, 'epochs': 200}
3 {'gen_x_times': 1.0, 'pregeneration_frac': 2, 'is_post_process': True, 'bot_filter_quantile': 0.001, 'top_filter_quantile': 0.999, 'use_adversarial': False, 'batch_size': 120, 'patience': 20, 'epochs': 100}
4 {'gen_x_times': 1.0, 'pregeneration_frac': 2, 'is_post_process': True, 'bot_filter_quantile': 0.001, 'top_filter_quantile': 0.999, 'use_adversarial': False, 'batch_size': 120, 'patience': 20, 'epochs': 200}
5 {'gen_x_times': 1.0, 'pregeneration_frac': 2, 'is_post_process': True, 'bot_filter_quantile': 0.001, 'top_filter_quantile': 0.999, 'u

In [8]:
# Cell 6: helper functions for scoring synthetic data against validation data

numeric_cols = NUMERIC_COLS
categorical_cols = CATEGORICAL_COLS

def safe_prob(series):
    return series.value_counts(normalize=True).sort_index()

def categorical_distribution_distance(real_df, synth_df, col):
    real_probs = safe_prob(real_df[col])
    synth_probs = safe_prob(synth_df[col])

    all_values = sorted(set(real_probs.index).union(set(synth_probs.index)))
    real_vec = np.array([real_probs.get(x, 0.0) for x in all_values], dtype=float)
    synth_vec = np.array([synth_probs.get(x, 0.0) for x in all_values], dtype=float)

    return float(np.mean(np.abs(real_vec - synth_vec)))

def numeric_distribution_distance(real_df, synth_df, cols):
    scores = []

    for col in cols:
        real_mean = real_df[col].mean()
        synth_mean = synth_df[col].mean()
        real_std = real_df[col].std(ddof=0)
        synth_std = synth_df[col].std(ddof=0)

        mean_den = max(abs(real_mean), 1e-8)
        std_den = max(abs(real_std), 1e-8)

        mean_score = abs(real_mean - synth_mean) / mean_den
        std_score = abs(real_std - synth_std) / std_den

        scores.append(mean_score)
        scores.append(std_score)

    return float(np.mean(scores))

def score_synthetic_vs_validation(real_val_df, synth_df):
    num_score = numeric_distribution_distance(real_val_df, synth_df, numeric_cols)

    cat_scores = [
        categorical_distribution_distance(real_val_df, synth_df, col)
        for col in categorical_cols
    ]
    cat_score = float(np.mean(cat_scores)) if cat_scores else 0.0

    total_score = 0.8 * num_score + 0.2 * cat_score

    return {
        "numeric_score": num_score,
        "categorical_score": cat_score,
        "total_score": total_score
    }

In [9]:
# Cell 7: helper to build a readable run name with stage information

def build_run_name(cfg, run_idx, stage_name):
    return (
        f"{stage_name}_run_{run_idx:03d}"
        f"_gxt{cfg['gen_x_times']}"
        f"_preg{cfg['pregeneration_frac']}"
        f"_post{int(cfg['is_post_process'])}"
        f"_adv{int(cfg['use_adversarial'])}"
        f"_bs{cfg['batch_size']}"
        f"_pat{cfg['patience']}"
        f"_ep{cfg['epochs']}"
        f"_bq{cfg['bot_filter_quantile']}"
        f"_tq{cfg['top_filter_quantile']}"
    )

In [12]:
# Cell 8: single grid-search run for one configuration with downstream TSTR evaluation

feature_cols = NUMERIC_COLS + CATEGORICAL_COLS
cat_cols = CATEGORICAL_COLS

def train_tstr_classifier(
    synth_train_df,
    real_val_df,
    real_test_df,
    max_epochs=60,
    patience=5,
    lr=1e-3,
    batch_size=256,
    hidden_dims=(64, 32),
    dropout=0.1
):
    feature_cols = NUMERIC_COLS + CATEGORICAL_COLS

    X_train_num = synth_train_df[NUMERIC_COLS].copy()
    X_val_num = real_val_df[NUMERIC_COLS].copy()
    X_test_num = real_test_df[NUMERIC_COLS].copy()

    X_train_cat = synth_train_df[CATEGORICAL_COLS].astype(str).copy()
    X_val_cat = real_val_df[CATEGORICAL_COLS].astype(str).copy()
    X_test_cat = real_test_df[CATEGORICAL_COLS].astype(str).copy()

    y_train_raw = synth_train_df[TARGET_COL].astype(str).copy()
    y_val_raw = real_val_df[TARGET_COL].astype(str).copy()
    y_test_raw = real_test_df[TARGET_COL].astype(str).copy()

    scaler = StandardScaler()
    X_train_num_scaled = scaler.fit_transform(X_train_num)
    X_val_num_scaled = scaler.transform(X_val_num)
    X_test_num_scaled = scaler.transform(X_test_num)

    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X_train_cat_enc = ohe.fit_transform(X_train_cat)
    X_val_cat_enc = ohe.transform(X_val_cat)
    X_test_cat_enc = ohe.transform(X_test_cat)

    X_train = np.hstack([X_train_num_scaled, X_train_cat_enc]).astype(np.float32)
    X_val = np.hstack([X_val_num_scaled, X_val_cat_enc]).astype(np.float32)
    X_test = np.hstack([X_test_num_scaled, X_test_cat_enc]).astype(np.float32)

    label_enc = LabelEncoder()
    y_train = label_enc.fit_transform(y_train_raw)
    y_val = label_enc.transform(y_val_raw)
    y_test = label_enc.transform(y_test_raw)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=batch_size,
        shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(X_test_tensor, y_test_tensor),
        batch_size=batch_size,
        shuffle=False
    )

    input_dim = X_train.shape[1]
    num_classes = len(label_enc.classes_)

    layers = []
    prev_dim = input_dim
    for hidden_dim in hidden_dims:
        layers.append(nn.Linear(prev_dim, hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        prev_dim = hidden_dim
    layers.append(nn.Linear(prev_dim, num_classes))

    model = nn.Sequential(*layers).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    best_epoch = 0
    best_state_dict = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())

        mean_val_loss = float(np.mean(val_losses))

        if mean_val_loss < best_val_loss:
            best_val_loss = mean_val_loss
            best_epoch = epoch
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state_dict is None:
        raise ValueError("No best model state was saved during TSTR training.")

    model.load_state_dict(best_state_dict)
    model.to(DEVICE)
    model.eval()

    test_losses = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = criterion(logits, yb)
            preds = torch.argmax(logits, dim=1)

            test_losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(yb.cpu().numpy())

    test_loss = float(np.mean(test_losses))
    macro_f1 = float(f1_score(all_targets, all_preds, average="macro"))

    return {
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "test_loss": test_loss,
        "macro_f1": macro_f1
    }

def run_single_tabgan_config(cfg, run_idx, stage_name):
    run_name = build_run_name(cfg, run_idx, stage_name)
    print(f"\n{'=' * 80}")
    print(f"Running configuration: {run_name}")
    print(cfg)
    print(f"{'=' * 80}")

    class_outputs = []
    summary_rows = []

    for attack_label, n_target in target_counts.items():
        print(f"\nTraining separate TabGAN for class: {attack_label}")

        train_class_df = train_df_global[train_df_global[TARGET_COL] == attack_label].copy()
        val_class_df = val_df_global[val_df_global[TARGET_COL] == attack_label].copy()

        if train_class_df.empty:
            raise ValueError(f"No training rows found for class '{attack_label}'.")

        if len(train_class_df) < 10:
            raise ValueError(f"Not enough training rows for class '{attack_label}' to train a stable TabGAN.")

        train_class_df["Attack_enc"] = 0

        train_df = train_class_df[feature_cols].copy()
        target_df = train_class_df[["Attack_enc"]].copy()
        reference_df = val_class_df[feature_cols].copy()

        generator = GANGenerator(
            gen_x_times=cfg["gen_x_times"],
            cat_cols=cat_cols,
            pregeneration_frac=cfg["pregeneration_frac"],
            is_post_process=cfg["is_post_process"],
            bot_filter_quantile=cfg["bot_filter_quantile"],
            top_filter_quantile=cfg["top_filter_quantile"],
            gen_params={
                "batch_size": cfg["batch_size"],
                "patience": cfg["patience"],
                "epochs": cfg["epochs"]
            }
        )

        synthetic_x, synthetic_y = generator.generate_data_pipe(
            train_df=train_df,
            target=target_df,
            test_df=reference_df,
            deep_copy=True,
            only_adversarial=False,
            use_adversarial=cfg["use_adversarial"]
        )

        synthetic_df = pd.concat(
            [synthetic_x.reset_index(drop=True), synthetic_y.reset_index(drop=True)],
            axis=1
        )

        if "Attack_enc" not in synthetic_df.columns and synthetic_df.shape[1] == len(feature_cols) + 1:
            synthetic_df.columns = feature_cols + ["Attack_enc"]

        synthetic_df[TARGET_COL] = attack_label
        synthetic_df = synthetic_df[feature_cols + [TARGET_COL]].copy()

        if synthetic_df.empty:
            raise ValueError(f"TabGAN produced no rows for class '{attack_label}' in run '{run_name}'.")

        score_dict = score_synthetic_vs_validation(
            val_class_df[feature_cols + [TARGET_COL]],
            synthetic_df
        )

        synthetic_resampled_df = synthetic_df.sample(
            n=n_target,
            replace=len(synthetic_df) < n_target,
            random_state=SEED
        ).reset_index(drop=True)

        class_output_path = SYNTHETIC_DIR / f"{run_name}_{attack_label}.csv"
        synthetic_resampled_df.to_csv(class_output_path, index=False)

        class_outputs.append(synthetic_resampled_df)

        summary_rows.append({
            "stage": stage_name,
            "run_name": run_name,
            "class": attack_label,
            "requested_rows": n_target,
            "raw_generated_rows": len(synthetic_df),
            "saved_rows": len(synthetic_resampled_df),
            "numeric_score": score_dict["numeric_score"],
            "categorical_score": score_dict["categorical_score"],
            "total_score": score_dict["total_score"],
            "class_output_path": str(class_output_path),
            **cfg
        })

    merged_df = pd.concat(class_outputs, ignore_index=True)
    merged_df = merged_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    merged_output_path = SYNTHETIC_DIR / f"{run_name}_merged.csv"
    merged_df.to_csv(merged_output_path, index=False)

    summary_df = pd.DataFrame(summary_rows)
    run_score = summary_df["total_score"].mean()

    tstr_metrics = train_tstr_classifier(
        synth_train_df=merged_df,
        real_val_df=val_df_global[feature_cols + [TARGET_COL]],
        real_test_df=test_df_global[feature_cols + [TARGET_COL]],
        max_epochs=60,
        patience=5,
        lr=1e-3,
        batch_size=256,
        hidden_dims=(64, 32),
        dropout=0.1
    )

    run_result = {
        "stage": stage_name,
        "run_name": run_name,
        "mean_total_score": run_score,
        "merged_output_path": str(merged_output_path),
        "best_epoch": tstr_metrics["best_epoch"],
        "best_val_loss": tstr_metrics["best_val_loss"],
        "test_loss": tstr_metrics["test_loss"],
        "macro_f1": tstr_metrics["macro_f1"],
        **cfg
    }

    return run_result, summary_df

In [ ]:
# Cell 9: execute two-stage coarse-to-fine grid search

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

def build_stage2_param_grid(best_cfg):
    stage2_param_grid = {
        "gen_x_times": sorted(set([
            max(0.5, best_cfg["gen_x_times"] - 0.25),
            best_cfg["gen_x_times"],
            best_cfg["gen_x_times"] + 0.25
        ])),
        "pregeneration_frac": sorted(set([
            max(1, best_cfg["pregeneration_frac"] - 1),
            best_cfg["pregeneration_frac"],
            best_cfg["pregeneration_frac"] + 1
        ])),
        "is_post_process": [best_cfg["is_post_process"]],
        "bot_filter_quantile": sorted(set([
            max(0.0, best_cfg["bot_filter_quantile"] / 10),
            best_cfg["bot_filter_quantile"],
            min(0.01, best_cfg["bot_filter_quantile"] * 10)
        ])),
        "top_filter_quantile": sorted(set([
            max(0.99, best_cfg["top_filter_quantile"] - 0.001),
            best_cfg["top_filter_quantile"],
            min(0.9999, best_cfg["top_filter_quantile"] + 0.001)
        ])),
        "use_adversarial": [best_cfg["use_adversarial"]],
        "batch_size": sorted(set([
            max(32, best_cfg["batch_size"] - 50),
            best_cfg["batch_size"],
            best_cfg["batch_size"] + 50
        ])),
        "patience": sorted(set([
            max(5, best_cfg["patience"] - 5),
            best_cfg["patience"],
            best_cfg["patience"] + 5
        ])),
        "epochs": sorted(set([
            max(50, best_cfg["epochs"] - 50),
            best_cfg["epochs"],
            best_cfg["epochs"] + 50
        ]))
    }
    return stage2_param_grid

def run_stage(configs, stage_name):
    stage_run_results = []
    stage_class_results = []

    run_results_csv_path = OUTPUT_DIR / f"{stage_name}_run_results.csv"
    class_results_csv_path = OUTPUT_DIR / f"{stage_name}_class_results.csv"

    if run_results_csv_path.exists():
        run_results_csv_path.unlink()

    if class_results_csv_path.exists():
        class_results_csv_path.unlink()

    print(f"\n{stage_name} run results CSV: {run_results_csv_path.resolve()}")
    print(f"{stage_name} class results CSV: {class_results_csv_path.resolve()}")

    completed_runs_by_signature = {}

    for run_idx, cfg in enumerate(configs, start=1):
        signature = tuple(sorted({
            k: v for k, v in cfg.items() if k != "epochs"
        }.items()))

        current_epochs = cfg["epochs"]

        if current_epochs > 100:
            prior_result = completed_runs_by_signature.get(signature)
            if prior_result is not None:
                prior_epochs = prior_result.get("epochs")
                prior_best_epoch = prior_result.get("best_epoch")

                if (
                    prior_epochs == 100
                    and prior_best_epoch is not None
                    and prior_best_epoch <= 90
                ):
                    print(
                        f"Skipping {stage_name} run {run_idx} because matching config with "
                        f"epochs=100 already reached best_epoch={prior_best_epoch} (<= 90)."
                    )
                    continue

        try:
            run_result, class_summary_df = run_single_tabgan_config(cfg, run_idx, stage_name)

            run_result = {
                "stage": stage_name,
                **cfg,
                **run_result
            }

            stage_run_results.append(run_result)
            completed_runs_by_signature[signature] = run_result

            run_row_df = pd.DataFrame([run_result])
            run_row_df.to_csv(
                run_results_csv_path,
                mode="a",
                header=not run_results_csv_path.exists(),
                index=False
            )

            if class_summary_df is not None and not class_summary_df.empty:
                class_summary_df = class_summary_df.copy()
                if "stage" not in class_summary_df.columns:
                    class_summary_df["stage"] = stage_name
                if "run_name" not in class_summary_df.columns:
                    class_summary_df["run_name"] = run_result.get(
                        "run_name",
                        f"{stage_name}_run_{run_idx:03d}"
                    )

                stage_class_results.append(class_summary_df)

                class_summary_df.to_csv(
                    class_results_csv_path,
                    mode="a",
                    header=not class_results_csv_path.exists(),
                    index=False
                )

            print(
                f"{stage_name} run {run_idx} saved -> "
                f"{run_results_csv_path.resolve()} | exists={run_results_csv_path.exists()}"
            )

        except Exception as e:
            error_row = pd.DataFrame([{
                "stage": stage_name,
                "run_name": f"{stage_name}_run_{run_idx:03d}",
                "error": str(e),
                **cfg
            }])
            error_path = OUTPUT_DIR / f"{stage_name}_error_run_{run_idx:03d}.csv"
            error_row.to_csv(error_path, index=False)
            print(f"{stage_name} run {run_idx} failed: {e}")
            print(f"Error CSV written to: {error_path.resolve()}")

    if run_results_csv_path.exists():
        stage_run_results_df = pd.read_csv(run_results_csv_path).sort_values(
            ["best_val_loss", "mean_total_score"],
            ascending=[True, True]
        ).reset_index(drop=True)
    else:
        stage_run_results_df = pd.DataFrame()

    if class_results_csv_path.exists():
        stage_class_results_df = pd.read_csv(class_results_csv_path)
    else:
        stage_class_results_df = pd.DataFrame()

    return stage_run_results_df, stage_class_results_df

print("\nStarting Stage 1 coarse search...")
stage1_run_results_df, stage1_class_results_df = run_stage(stage1_configs, "stage1")

display(stage1_run_results_df.head(10))
display(stage1_class_results_df.head(10))

if stage1_run_results_df.empty:
    raise ValueError("Stage 1 produced no successful runs, so Stage 2 cannot be started.")

best_stage1 = stage1_run_results_df.iloc[0].to_dict()

print("\nBest Stage 1 configuration:")
print(json.dumps(best_stage1, indent=2, default=str))

stage2_param_grid = build_stage2_param_grid(best_stage1)
stage2_configs = expand_param_grid(stage2_param_grid)

print("\nStage 2 fine grid preview:")
for i, cfg in enumerate(stage2_configs[:5], start=1):
    print(i, cfg)

print("Total Stage 2 configurations:", len(stage2_configs))

print("\nStarting Stage 2 fine search...")
stage2_run_results_df, stage2_class_results_df = run_stage(stage2_configs, "stage2")

display(stage2_run_results_df.head(10))
display(stage2_class_results_df.head(10))

all_run_results_df = pd.concat(
    [stage1_run_results_df, stage2_run_results_df],
    ignore_index=True
).sort_values(
    ["best_val_loss", "mean_total_score"],
    ascending=[True, True]
).reset_index(drop=True)

all_class_results_df = pd.concat(
    [stage1_class_results_df, stage2_class_results_df],
    ignore_index=True
)

all_run_results_path = OUTPUT_DIR / "tabgan_gridsearch_all_run_results.csv"
all_class_results_path = OUTPUT_DIR / "tabgan_gridsearch_all_class_results.csv"

all_run_results_df.to_csv(all_run_results_path, index=False)
all_class_results_df.to_csv(all_class_results_path, index=False)

print("\nFinal combined run results CSV:", all_run_results_path.resolve())
print("Final combined class results CSV:", all_class_results_path.resolve())

display(all_run_results_df.head(10))
display(all_class_results_df.head(10))

OUTPUT_DIR: /data/home/hay37276/gridSearchProject_YannicHanel/gridsearch_runs/output

Starting Stage 1 coarse search...

stage1 run results CSV: /data/home/hay37276/gridSearchProject_YannicHanel/gridsearch_runs/output/stage1_run_results.csv
stage1 class results CSV: /data/home/hay37276/gridSearchProject_YannicHanel/gridsearch_runs/output/stage1_class_results.csv

Running configuration: stage1_run_001_gxt1.0_preg2_post1_adv0_bs120_pat10_ep100_bq0.001_tq0.999
{'gen_x_times': 1.0, 'pregeneration_frac': 2, 'is_post_process': True, 'bot_filter_quantile': 0.001, 'top_filter_quantile': 0.999, 'use_adversarial': False, 'batch_size': 120, 'patience': 10, 'epochs': 100}

Training separate TabGAN for class: Backdoor


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/100 [00:00<?, ?it/s]


Training separate TabGAN for class: none


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

...

Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/50 [00:00<?, ?it/s]

stage2 run 52 saved -> /data/home/hay37276/gridSearchProject_YannicHanel/gridsearch_runs/output/stage2_run_results.csv | exists=True

Running configuration: stage2_run_053_gxt0.75_preg1_post1_adv0_bs300_pat25_ep100_bq0.0001_tq0.999
{'gen_x_times': 0.75, 'pregeneration_frac': 1, 'is_post_process': True, 'bot_filter_quantile': 0.0001, 'top_filter_quantile': 0.999, 'use_adversarial': False, 'batch_size': 300, 'patience': 25, 'epochs': 100}

Training separate TabGAN for class: Backdoor


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# Cell 10: inspect best TabGAN configuration

best_tabgan = run_results_df.iloc[0].to_dict()

print("Best TabGAN configuration:")
print(json.dumps(best_tabgan, indent=2, default=str))

metadata = {
    "seed": SEED,
    "data_path": DATA_PATH,
    "device": DEVICE,
    "train_rows": len(train_df_global),
    "val_rows": len(val_df_global),
    "test_rows": len(test_df_global),
    "num_configs": len(stage1_configs),
    "target_counts": target_counts,
    "target_distribution": target_distribution,
    "numeric_cols": NUMERIC_COLS,
    "categorical_cols": CATEGORICAL_COLS,
    "target_col": TARGET_COL
}

In [ ]:
with open(OUTPUT_DIR / "tabgan_gridsearch_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print("-", OUTPUT_DIR / "tabgan_gridsearch_run_results.csv")
print("-", OUTPUT_DIR / "tabgan_gridsearch_class_results.csv")
print("-", OUTPUT_DIR / "tabgan_gridsearch_metadata.json")
print("-", SYNTHETIC_DIR)